# Setup

In [ ]:
# Tìm thư mục gốc của dự án để notebook biết nơi chứa mã nguồn và dữ liệu.
from pathlib import Path
import sys
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from PIL import Image

def find_project_root(max_up=6):
    p = Path.cwd()
    for _ in range(max_up + 1):
        if (p / "src").exists() and (p / "configs").exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    return Path.cwd()

PROJECT_ROOT = find_project_root()

sys.path.append(str(PROJECT_ROOT))

from src.config import load_yaml
from src.io_utils import save_csv, save_numpy

pipeline_cfg = load_yaml(PROJECT_ROOT / "configs/pipeline_config.yaml")
reid_cfg = pipeline_cfg["reid"]

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)


In [ ]:
# Cell này thực hiện một bước cụ thể trong luồng pipeline, có thể là nạp dữ liệu, xử lý, hoặc lưu kết quả.
# !pip install git+https://github.com/KaiyangZhou/deep-person-reid.git


In [ ]:
# Cell này thực hiện một bước cụ thể trong luồng pipeline, có thể là nạp dữ liệu, xử lý, hoặc lưu kết quả.
!pip install torchreid


In [ ]:
# Cell này thực hiện một bước cụ thể trong luồng pipeline, có thể là nạp dữ liệu, xử lý, hoặc lưu kết quả.
import sys
print(sys.executable)
import sys
!"{sys.executable}" -m pip install tensorboard
import sys
!"{sys.executable}" -m pip install tensorboard yacs gdown


In [ ]:
# Nạp mô hình ReID để trích đặc trưng từ ảnh nhân vật, sử dụng cho việc ghép người giữa camera.
import torchreid
print("torchreid imported successfully")


# Load OSNet

In [ ]:
# Nạp mô hình ReID để trích đặc trưng từ ảnh nhân vật, sử dụng cho việc ghép người giữa camera.
import torchreid

reid_model = torchreid.models.build_model(
    name=reid_cfg["model"],
    num_classes=1000,
    pretrained=True
)

reid_model = reid_model.to(device)
reid_model.eval()

print("Loaded Re-ID model:", reid_cfg["model"])


# Transform ảnh

In [ ]:
# Nạp mô hình ReID để trích đặc trưng từ ảnh nhân vật, sử dụng cho việc ghép người giữa camera.
from torchvision import transforms

input_h = reid_cfg["input_height"]
input_w = reid_cfg["input_width"]

transform = transforms.Compose([
    transforms.Resize((input_h, input_w)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


def extract_embedding(image_path):
    img = Image.open(image_path).convert("RGB")
    tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        feat = reid_model(tensor)
        feat = torch.nn.functional.normalize(feat, p=2, dim=1)

    return feat.cpu().numpy()[0]


# Extract crop embeddings

In [ ]:
# Cell này thực hiện một bước cụ thể trong luồng pipeline, có thể là nạp dữ liệu, xử lý, hoặc lưu kết quả.
rep_df = pd.read_csv(PROJECT_ROOT / "outputs/crops/representative_crops.csv")

embedding_rows = []
embeddings = []

for idx, row in tqdm(rep_df.iterrows(), total=len(rep_df)):
    emb = extract_embedding(row["crop_path"])
    embeddings.append(emb)

    embedding_rows.append({
        "embedding_index": idx,
        "camera_id": row["camera_id"],
        "local_track_id": int(row["local_track_id"]),
        "frame_id": int(row["frame_id"]),
        "timestamp_sec": float(row["timestamp_sec"]),
        "crop_path": row["crop_path"],
        "quality_score": float(row["quality_score"])
    })

embeddings = np.vstack(embeddings)
emb_meta_df = pd.DataFrame(embedding_rows)

save_numpy(embeddings, PROJECT_ROOT / "outputs/embeddings/crop_embeddings.npy")
save_csv(emb_meta_df, PROJECT_ROOT / "outputs/embeddings/crop_embeddings_meta.csv")

print("Embeddings shape:", embeddings.shape)
emb_meta_df.head()


# Tracklet-level embedding

In [ ]:
# Cell này thực hiện một bước cụ thể trong luồng pipeline, có thể là nạp dữ liệu, xử lý, hoặc lưu kết quả.
emb_meta_df = pd.read_csv(PROJECT_ROOT / "outputs/embeddings/crop_embeddings_meta.csv")
embeddings = np.load(PROJECT_ROOT / "outputs/embeddings/crop_embeddings.npy")

tracklet_emb_rows = []
tracklet_embs = []

for (camera_id, track_id), group in emb_meta_df.groupby(["camera_id", "local_track_id"]):
    idxs = group["embedding_index"].values
    embs = embeddings[idxs]

    mean_emb = embs.mean(axis=0)
    mean_emb = mean_emb / np.linalg.norm(mean_emb)

    emb_index = len(tracklet_embs)
    tracklet_embs.append(mean_emb)

    tracklet_emb_rows.append({
        "tracklet_embedding_index": emb_index,
        "camera_id": camera_id,
        "local_track_id": int(track_id),
        "num_crops": len(group),
        "mean_quality_score": float(group["quality_score"].mean())
    })

tracklet_embs = np.vstack(tracklet_embs)
tracklet_emb_df = pd.DataFrame(tracklet_emb_rows)

save_numpy(tracklet_embs, PROJECT_ROOT / "outputs/embeddings/tracklet_embeddings.npy")
save_csv(tracklet_emb_df, PROJECT_ROOT / "outputs/embeddings/tracklet_embeddings_meta.csv")

print("Tracklet embeddings:", tracklet_embs.shape)
tracklet_emb_df.head()


# Some check

In [ ]:
# Cell này thực hiện một bước cụ thể trong luồng pipeline, có thể là nạp dữ liệu, xử lý, hoặc lưu kết quả.
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

embedding_files = []
embedding_files += list((PROJECT_ROOT / "outputs").rglob("*embedding*.csv"))
embedding_files += list((PROJECT_ROOT / "outputs").rglob("*embedding*.npy"))
embedding_files += list((PROJECT_ROOT / "outputs").rglob("*embeddings*.csv"))
embedding_files += list((PROJECT_ROOT / "outputs").rglob("*embeddings*.npy"))

print("Embedding files found:")
for p in embedding_files:
    print(p)

for p in embedding_files:
    print("\nFILE:", p)
    if p.suffix == ".csv":
        df = pd.read_csv(p)
        print("Rows:", len(df))
        print("Columns:", list(df.columns)[:20])
        print(df.head())
    elif p.suffix == ".npy":
        arr = np.load(p, allow_pickle=True)
        print("Shape:", arr.shape)
        print("Dtype:", arr.dtype)
